# Simple Molecular Dynamics Simulation
Use harmonic oscillator as an example

by Min-Yeh Tsai (2024/4/24)

In [ ]:
# Import
import math
import numpy as np

# I. Anylytical solution

The Hamiltonian $H(q,p)$ for a harmonic oscillator is $H = \mbox{kinetic energy} + \mbox{potential energy} = \frac{p^2}{2m} + \frac{1}{2}kq^2 $,

where $m$ is the mass, $k$ is the force constant, and that $E$ is the total energy.

Since the equations of motion for a harmonic oscillator, $q(t)$, follows a periodic motion, we can define its angular frequency ($\omega$) and the period ($T=\frac{1}{\nu}$):

$\omega=\sqrt{\frac{k}{m}}=2\pi\nu$ and that $\nu = \frac{\omega}{2\pi}$


The parameters described above can be implemented into `python` through the step called *declaration* shown in the code block below.

In [ ]:
# Parameters
mass = 2 # m
force_const = 9 # k
energy = 1.5 # E
delta_t = 0.3 # dt
t_min = 0
t_max = 10
phase = 0 # phi

angular_freq = math.sqrt(force_const/mass) # omega
freq = angular_freq/(2 * math.pi) # nu
period = 1/freq # T
amplitude = math.sqrt(2 * energy / force_const) # A

args = {'m':mass,'E':energy,'k':force_const,'nu':freq,'omega':angular_freq,'A':amplitude,'phi':phase}

In [ ]:
print(angular_freq,"\n\n", freq,"\n\n",period)

In [ ]:
t = np.arange(t_min,t_max,delta_t)
print(t)

In [ ]:
# Function definition
def coordinate(t,args):
    """
    general coordinate
    """

    amplitude = args['A']
    omega = args['omega']
    phase = args['phi']

    q = amplitude * np.cos(omega * t + phase)
    return q

def momentum(t,args):
    """
    momentum
    """

    mass = args['m']
    amplitude = args['A']
    omega = args['omega']
    phase = args['phi']

    p = -mass * omega * amplitude * np.sin(omega * t + phase)
    return p

def hamiltonian(p,q,args):
    """
    Total energy (Hamiltonian)
    """

    mass = args['m']
    force_const = args['k']

    h = pow(p,2) / (2 * mass) + 0.5 * force_const * pow(q,2)
    return h

In [ ]:
q = coordinate(t,args)
print(q)

In [ ]:
p = momentum(t,args)
print(p)

In [ ]:
h = hamiltonian(p,q,args)
print(h)

## Plot

In [ ]:
import matplotlib.pyplot as plt
plt.plot(t,h)
plt.show()

## Animation

In [ ]:
# Import
%matplotlib inline
#import numpy as np
#import matplotlib.pyplot as plt

from matplotlib import animation, rc
from IPython.display import HTML, Image

### 1. Position vs time

In [ ]:
#setup figure 1
fig1 = plt.figure()
ax1 = plt.axes(xlim=(0, 10), ylim=(-1, 1))
line, = ax1.plot([], [], marker = ".")
t_text = ax1.text(0.5, 0.75, '', fontsize=18)
ax1.set_xlabel('Time (s)', fontsize = 16)
ax1.set_ylabel('Position (q)', fontsize = 16)

In [ ]:
# initialization function: plot the background of each frame
def init():
    line.set_data([], [])
    t_text.set_text('t=')
    return line,

# animation function.  This is called sequentially
# def animate(i):
#     tt = t[i]
#     qq = coordinate(t[i],args)
#     line.set_data(tt, qq)
#     #print(tt,qq)
#     t_text.set_text('t = %3.2f' % tt)
#     return line,

# Modified animation function
def animate(i):
    tt = t[:i+1] # Get time values up to the current frame i
    qq = coordinate(t[:i+1],args) # Get corresponding position values up to the current frame i
    line.set_data(tt, qq)
    #print(tt,qq)
    t_text.set_text('t = %3.2f' % t[i]) # Display the time at the current frame
    return line,

In [ ]:
# call the animator.  blit=True means only re-draw the parts that have changed.
anim = animation.FuncAnimation(fig1, animate, init_func=init,
                               frames=34, interval=1000, blit=True)
rc('animation', html='jshtml')
anim

### 2. Position vs momentum

In [ ]:
#setup figure 2
fig2 = plt.figure()
ax2 = plt.axes(xlim=(-3, 3), ylim=(-1, 1))
line2, = ax2.plot([], [], marker = ".")
t_text_2 = ax2.text(-2.5, 0.75, '', fontsize=18)
ax2.set_xlabel('Momentum (p)', fontsize = 16)
ax2.set_ylabel('Position (q)', fontsize = 16)

In [ ]:
# initialization function: plot the background of each frame
def init_2():
    line2.set_data([], [])
    t_text_2.set_text('t=')
    return line2,

# animation function.  This is called sequentially
# def animate_2(i):
#     tt = t[i]
#     qq = coordinate(t[i],args)
#     pp = momentum(t[i],args)
#     line2.set_data(pp, qq)
#     #print(tt,qq)
#     t_text_2.set_text('t = %3.2f' % tt)
#     return line2,

# Modified animation function
def animate_2(i):
    qq_seq = coordinate(t[:i+1], args) # Get position values up to the current frame i
    pp_seq = momentum(t[:i+1], args) # Get momentum values up to the current frame i
    line2.set_data(pp_seq, qq_seq)
    t_text_2.set_text('t = %3.2f' % t[i]) # Display the time at the current frame
    return line2,

In [ ]:
# call the animator.  blit=True means only re-draw the parts that have changed.
anim_2 = animation.FuncAnimation(fig2, animate_2, init_func=init_2,
                               frames=34, interval=1000, blit=True)
rc('animation', html='jshtml')
anim_2

# II. Euler method

Using taylor expansion with respect to $\Delta t$, we have

$q_i(t+\Delta t)=q_i(t)+\dot{q_i}(t)\Delta t=q_i(t)+\left[p_i(t)/m\right]\Delta t$ (for position propagation) Eq.(2.1)

$p_i(t+\Delta t)=p_i(t)+\dot{p_i}(t)\Delta t=p_i(t)-kq_i(t)\Delta t$ (for momentum propagation) Eq. (2.2)

where $\dot{q(t)}=\frac{\partial H}{\partial p}=\frac{p(t)}{m}$ and $\dot{p(t)}=-\frac{\partial H}{\partial q}=-kq(t)$ are used.

**IMPORTANT!!**

Euler method employs a mathematical expansion of a function, so each function is solved with a numerical uncertainty $\Delta t$. Thus, the equation of motion is obtained through many iterative processes, which is differernt from the analytical solution described above. The deviation due to $\Delta t$ will be accumulated throughout the whole process. Please be aware of the differernce!!

In [ ]:
# Set initial conditions
q0 = coordinate(0,args) # see Function definition in I.
p0 = momentum(0,args) # see Function definition in I.
print("q(t=0) =", q0,"\n")
print("p(t=0) =", p0)

In [ ]:
# Function definition
def q_dot(p, args):
    """
    Dot of coordinate
    """
    mass = args['m']
    qdot = p / mass
    return qdot

def p_dot(q, args):
    """
    Dot of momentum
    """
    force_const = args['k']
    pdot = -force_const * q
    return pdot

def euler(q0,p0,delta_t,n_iter):
    """
    Euler method
    q0: initial position
    p0: initial momentum
    delta_t: time differential
    n_iter: number of iteration
    """
    q = q0
    p = p0
    t_ = [0]
    q_ = [q0]
    p_ = [p0]
    #Iteration
    for i in range(n_iter):
        qq = q + q_dot(p,args) * delta_t # q(t+dt)=q(t)+qdot*dt
        pp = p + p_dot(q,args) * delta_t # p(t+dt)=p(t)+pdot*dt
        # Recording data
        t_ = np.append(t_, (i+1) * delta_t) # time array
        q_ = np.append(q_, qq) # coordinate array
        p_ = np.append(p_, pp) # momentum array
        q = qq # updating q for the next round
        p = pp # updating p for the next round
        #print(i,qq,pp)
    return t_, q_, p_


In [ ]:
q_dot(p0,args)

In [ ]:
p_dot(q0,args)

In [ ]:
# Distribute the output
t_euler, q_euler, p_euler = euler(q0,p0,delta_t,30)

print(t_euler)
print(q_euler)
print(p_euler)

In [ ]:
h_euler = hamiltonian(p_euler,q_euler,args)
print(h_euler)

## Plot

In [ ]:
plt.plot(t_euler,h_euler)
plt.xlim([0, 5])
plt.ylim([1.3, 5])
plt.show()

## Animation

### 1. Position vs time

In [ ]:
#setup figure 3
fig3 = plt.figure()
ax3 = plt.axes(xlim=(0, 10), ylim=(-1, 1))
line, = ax3.plot([], [], marker = ".")
t_text = ax3.text(0.5, 0.75, '', fontsize=18)
ax3.set_xlabel('Time (s)', fontsize = 16)
ax3.set_ylabel('Position (q)', fontsize = 16)

In [ ]:
# call the animator.  blit=True means only re-draw the parts that have changed.
anim_3 = animation.FuncAnimation(fig3, animate, init_func=init,
                               frames=30, interval=1000, blit=True)
rc('animation', html='jshtml')
anim_3